In [1]:
import Pkg
Pkg.activate("..")
Pkg.status()

  Activating project at `~/quantum_computing/quantum_stochastic_programming`


Status `~/quantum_computing/quantum_stochastic_programming/Project.toml`
  [07493b3f] Alpine v0.5.8
  [336ed68f] CSV v0.10.16
  [a93c6f00] DataFrames v1.8.2
  [bb8be931] EAGO v0.9.2
  [87dc4568] HiGHS v1.23.0
  [b6b21f68] Ipopt v1.15.0
  [4076af6c] JuMP v1.30.1
  [2ddba703] Juniper v0.9.4
  [19f71287] MAiNGO v0.2.2
  [82193955] SCIP v0.12.8


In [2]:
import JuMP

# import Alpine
import EAGO
import HiGHS
import Ipopt
import Juniper
# import MAiNGO

In [3]:
import Random

import LinearAlgebra as LA

In [153]:
d = 8
# frac = 0.5

# ps = [0.1, 0.9]
# @assert(sum(ps) == 1.0)
# ns = length(ps)

# ws_max = [
#     0.0 1.0;
#     1.0 1.0;
#     0.0 1.0;
# ]

cx = [
    4.0 0.0;
    # 5.0 1e-2;
]
nx = size(cx, 1)
# xlb = zeros(nx)
xlb = fill(-Inf, nx)
# xub = fill(5.0, nx)
xub = fill(Inf, nx)

# cw = [
#     0.25 1e-3;
#     0.3  1e-3;
#     0.35 1e-3;
# ]
# nw = size(cw,1)
ny = 4
cy1 = range(2.0, 3.0, ny)
cy2 = fill(1.0, ny)
# cy1 = zeros(ny)
# cy2 = fill(4.0)
cy = hcat(cy1, cy2)
# ylb = zeros(ny)
ylb = fill(-Inf, ny)
# yub = fill(1.0, ny)
yub = fill(Inf, ny)

cr = 10.0

ns = 2^ny
ps = ones(ns) ./ ns
@assert(isapprox(sum(ps), 1.0))

# rng = Random.MersenneTwister(1234)
# xi = Random.rand(rng, ny, ns) .>= frac

xi = [((s-1) >> (ny-i)) & 1 for i in 1:ny, s in 1:ns]
# xi = zeros(ny, ns)
# idx = 0
# for k in 0:ny
#     @show binomial(ny, k)
#     for i in 1:binomial(ny, k)
#         idx += 1

#     end
# end
# @show idx

4×16 Matrix{Int64}:
 0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1
 0  0  0  0  1  1  1  1  0  0  0  0  1  1  1  1
 0  0  1  1  0  0  1  1  0  0  1  1  0  0  1  1
 0  1  0  1  0  1  0  1  0  1  0  1  0  1  0  1

In [154]:
#### Alpine ####
# ipopt = JuMP.optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0)
# highs = JuMP.optimizer_with_attributes(HiGHS.Optimizer, "output_flag" => false)
# solver = JuMP.optimizer_with_attributes(Alpine.Optimizer, "nlp_solver" => ipopt, "mip_solver" => highs)
#### EAGO ####
# solver = EAGO.Optimizer
#### HiGHS ####
# solver = HiGHS.Optimizer
#### Ipopt ####
solver = Ipopt.Optimizer
#### Juniper ####
# ipopt = JuMP.optimizer_with_attributes(Ipopt.Optimizer, "print_level"=>0)
# solver = JuMP.optimizer_with_attributes(Juniper.Optimizer, "nl_solver"=>ipopt)
#### MAiNGO ####
# solver = JuMP.optimizer_with_attributes(MAiNGO.Optimizer, "epsilonA"=> 1e-8)
m = JuMP.Model(solver)
# JuMP.@variable(m, x[1:nx] >= 0)
JuMP.@variable(m, xlb[i] <= x[i=1:nx] <= xub[i])
JuMP.@variable(m, ylb[i] <= y[i=1:ny, s=1:ns] <= yub[i])
# JuMP.@variable(m, ylb[i] <= y[i=1:ny, s=1:ns] <= yub[i], binary=true)

4×16 Matrix{JuMP.VariableRef}:
 y[1,1]  y[1,2]  y[1,3]  y[1,4]  y[1,5]  y[1,6]  …  y[1,12]  y[1,13]  y[1,14]  y[1,15]  y[1,16]
 y[2,1]  y[2,2]  y[2,3]  y[2,4]  y[2,5]  y[2,6]     y[2,12]  y[2,13]  y[2,14]  y[2,15]  y[2,16]
 y[3,1]  y[3,2]  y[3,3]  y[3,4]  y[3,5]  y[3,6]     y[3,12]  y[3,13]  y[3,14]  y[3,15]  y[3,16]
 y[4,1]  y[4,2]  y[4,3]  y[4,4]  y[4,5]  y[4,6]     y[4,12]  y[4,13]  y[4,14]  y[4,15]  y[4,16]

In [155]:
obj_fs = JuMP.@expression(
    m, obj_first_stage,
    sum(cx[i, 1] * x[i] + cx[i, 2] * x[i] * x[i] for i in 1:nx)
)

obj_fs_grad = JuMP.@expression(
    m, obj_first_stage_grad,
    sum(cx[i, 1] + cx[i, 2] * x[i] for i in 1:nx)
)

obj_ss = JuMP.@expression(
    m, obj_second_stage,
    sum(ps[s] * (cy[i, 1]*y[i, s] + cy[i, 2]*y[i, s]^2) * xi[i, s] for i in 1:ny, s in 1:ns)
    +
    sum(ps[s] * (cr * y[i, s]) * (1 - xi[i, s]) for i in 1:ny, s in 1:ns)
)

obj_ss_grad = JuMP.@expression(
    m, obj_second_stage_grad,
    1.0 / ny * sum(ps[s] * (cy[i, 1] + cy[i, 2]*y[i, s]) * xi[i, s] for i in 1:ny, s in 1:ns)
    +
    1.0 / ny * sum(ps[s] * cr * (1 - xi[i, s]) for i in 1:ny, s in 1:ns)
)

JuMP.@objective(m, Min, obj_fs + obj_ss)

# JuMP.@objective(m, Min, 
#     sum(cx[i,1] * x[i] + cx[i,2] * x[i] * x[i] for i in 1:nx)
#     + sum(ps[s] * (cy[i,1]*y[i,s] + cy[i,2]*y[i,s]^2) * xi[i,s] for i in 1:ny, s in 1:ns)
#     + sum(ps[s] * (cr * y[i,s]) * (1 - xi[i,s]) for i in 1:ny, s in 1:ns)
# )
# JuMP.@objective(m, Min, 
#     sum(cx[i,1] * x[i] for i in 1:nx)
#     + sum(ps[s] * cy[i,1] * y[i,s] * xi[i,s] for i in 1:ny, s in 1:ns)
#     + sum(ps[s] * cr * y[i,s] * (1 - xi[i,s]) for i in 1:ny, s in 1:ns)
# )

0.0625 y[1,9]² + 0.0625 y[1,10]² + 0.0625 y[1,11]² + 0.0625 y[1,12]² + 0.0625 y[1,13]² + 0.0625 y[1,14]² + 0.0625 y[1,15]² + 0.0625 y[1,16]² + 0.0625 y[2,5]² + 0.0625 y[2,6]² + 0.0625 y[2,7]² + 0.0625 y[2,8]² + 0.0625 y[2,13]² + 0.0625 y[2,14]² + 0.0625 y[2,15]² + 0.0625 y[2,16]² + 0.0625 y[3,3]² + 0.0625 y[3,4]² + 0.0625 y[3,7]² + 0.0625 y[3,8]² + 0.0625 y[3,11]² + 0.0625 y[3,12]² + 0.0625 y[3,15]² + 0.0625 y[3,16]² + 0.0625 y[4,2]² + 0.0625 y[4,4]² + 0.0625 y[4,6]² + 0.0625 y[4,8]² + 0.0625 y[4,10]² + 0.0625 y[4,12]² + 0.0625 y[4,14]² + 0.0625 y[4,16]² + 4 x[1] + 0.125 y[1,9] + 0.125 y[1,10] + 0.125 y[1,11] + 0.125 y[1,12] + 0.125 y[1,13] + 0.125 y[1,14] + 0.125 y[1,15] + 0.125 y[1,16] + 0.14583333333333334 y[2,5] + 0.14583333333333334 y[2,6] + 0.14583333333333334 y[2,7] + 0.14583333333333334 y[2,8] + 0.14583333333333334 y[2,13] + 0.14583333333333334 y[2,14] + 0.14583333333333334 y[2,15] + 0.14583333333333334 y[2,16] + 0.16666666666666666 y[3,3] + 0.16666666666666666 y[3,4] + 0.16666

In [156]:
for s in 1:ns
    JuMP.@constraint(m, sum(x[i] for i in 1:nx) + sum(y[i,s] for i in 1:ny) == d)
end

In [157]:
display(m)

A JuMP Model
├ solver: Ipopt
├ objective_sense: MIN_SENSE
│ └ objective_function_type: JuMP.QuadExpr
├ num_variables: 65
├ num_constraints: 16
│ └ JuMP.AffExpr in MOI.EqualTo{Float64}: 16
└ Names registered in the model
  └ :obj_first_stage, :obj_first_stage_grad, :obj_second_stage, :obj_second_stage_grad, :x, :y

In [158]:
JuMP.all_constraints(m, include_variable_in_set_constraints=true)

16-element Vector{JuMP.ConstraintRef}:
 x[1] + y[1,1] + y[2,1] + y[3,1] + y[4,1] = 8
 x[1] + y[1,2] + y[2,2] + y[3,2] + y[4,2] = 8
 x[1] + y[1,3] + y[2,3] + y[3,3] + y[4,3] = 8
 x[1] + y[1,4] + y[2,4] + y[3,4] + y[4,4] = 8
 x[1] + y[1,5] + y[2,5] + y[3,5] + y[4,5] = 8
 x[1] + y[1,6] + y[2,6] + y[3,6] + y[4,6] = 8
 x[1] + y[1,7] + y[2,7] + y[3,7] + y[4,7] = 8
 x[1] + y[1,8] + y[2,8] + y[3,8] + y[4,8] = 8
 x[1] + y[1,9] + y[2,9] + y[3,9] + y[4,9] = 8
 x[1] + y[1,10] + y[2,10] + y[3,10] + y[4,10] = 8
 x[1] + y[1,11] + y[2,11] + y[3,11] + y[4,11] = 8
 x[1] + y[1,12] + y[2,12] + y[3,12] + y[4,12] = 8
 x[1] + y[1,13] + y[2,13] + y[3,13] + y[4,13] = 8
 x[1] + y[1,14] + y[2,14] + y[3,14] + y[4,14] = 8
 x[1] + y[1,15] + y[2,15] + y[3,15] + y[4,15] = 8
 x[1] + y[1,16] + y[2,16] + y[3,16] + y[4,16] = 8

In [159]:
JuMP.optimize!(m)

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.1.

Number of nonzeros in equality constraint Jacobian...:       80
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:       32

Total number of variables............................:       65
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:       16
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 8.00e+00 4.72e-01  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [160]:
JuMP.solution_summary(m)

solution_summary(; result = 1, verbose = false)
├ solver_name          : Ipopt
├ Termination
│ ├ termination_status : LOCALLY_SOLVED
│ ├ result_count       : 1
│ └ raw_status         : Solve_Succeeded
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : -4.82194e+02
│ └ dual_objective_value : 3.20000e+01
└ Work counters
  ├ solve_time (sec)   : 2.56419e-03
  └ barrier_iterations : 4

In [161]:
JuMP.value(x)

1-element Vector{Float64}:
 184.9999997316214

In [162]:
JuMP.value(y)

4×16 Matrix{Float64}:
 -44.25  -60.1667  -60.2222   -92.0833   …    4.0         4.0         4.0      -44.0
 -44.25  -60.1667  -60.2222   -92.0833        3.83333     3.83333     3.83333  -44.1667
 -44.25  -60.1667    3.66667    3.66667     -92.4167   -188.333       3.66667  -44.3333
 -44.25    3.5     -60.2222     3.5         -92.4167      3.5      -188.5      -44.5

In [163]:
idx = abs.(JuMP.value(y) .- Int.(round.(JuMP.value(y)))) .> 1e-6
println("Number of fractional values: ", sum(idx))
;

Number of fractional values: 55


In [164]:
xi

4×16 Matrix{Int64}:
 0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1
 0  0  0  0  1  1  1  1  0  0  0  0  1  1  1  1
 0  0  1  1  0  0  1  1  0  0  1  1  0  0  1  1
 0  1  0  1  0  1  0  1  0  1  0  1  0  1  0  1

In [165]:
report = JuMP.primal_feasibility_report(m)
max_viol = isempty(report) ? 0.0 : maximum(values(report))

4.973799150320701e-14

In [166]:
@show JuMP.objective_value(m)
# @show sum(cx[:,1] .* JuMP.value(x))
# @show JuMP.objective_value(m) - sum(cx[:,1] .* JuMP.value(x))
@show sum(cx[:,1] .* JuMP.value(x) + cx[:,2] .* JuMP.value(x).^2)
@show JuMP.objective_value(m) - sum(cx[:,1] .* JuMP.value(x) + cx[:,2] .* JuMP.value(x).^2)
;

JuMP.objective_value(m) = -482.19444444444457
sum(cx[:, 1] .* JuMP.value(x) + cx[:, 2] .* JuMP.value(x) .^ 2) = 739.9999989264855
JuMP.objective_value(m) - sum(cx[:, 1] .* JuMP.value(x) + cx[:, 2] .* JuMP.value(x) .^ 2) = -1222.19444337093


In [167]:
nres = (xi - JuMP.value(y)) .< 0
@show sum(nres)
@show sum(nres) / (ny * ns)
;

sum(nres) = 28
sum(nres) / (ny * ns) = 0.4375


In [168]:
@show m[:obj_first_stage]
JuMP.value(m[:obj_first_stage])

m[:obj_first_stage] = 4 x[1]


739.9999989264855

In [169]:
@show m[:obj_second_stage]
JuMP.value(m[:obj_second_stage])

m[:obj_second_stage] = 0.0625 y[1,9]² + 0.0625 y[1,10]² + 0.0625 y[1,11]² + 0.0625 y[1,12]² + 0.0625 y[1,13]² + 0.0625 y[1,14]² + 0.0625 y[1,15]² + 0.0625 y[1,16]² + 0.0625 y[2,5]² + 0.0625 y[2,6]² + 0.0625 y[2,7]² + 0.0625 y[2,8]² + 0.0625 y[2,13]² + 0.0625 y[2,14]² + 0.0625 y[2,15]² + 0.0625 y[2,16]² + 0.0625 y[3,3]² + 0.0625 y[3,4]² + 0.0625 y[3,7]² + 0.0625 y[3,8]² + 0.0625 y[3,11]² + 0.0625 y[3,12]² + 0.0625 y[3,15]² + 0.0625 y[3,16]² + 0.0625 y[4,2]² + 0.0625 y[4,4]² + 0.0625 y[4,6]² + 0.0625 y[4,8]² + 0.0625 y[4,10]² + 0.0625 y[4,12]² + 0.0625 y[4,14]² + 0.0625 y[4,16]² + 0.125 y[1,9] + 0.125 y[1,10] + 0.125 y[1,11] + 0.125 y[1,12] + 0.125 y[1,13] + 0.125 y[1,14] + 0.125 y[1,15] + 0.125 y[1,16] + 0.14583333333333334 y[2,5] + 0.14583333333333334 y[2,6] + 0.14583333333333334 y[2,7] + 0.14583333333333334 y[2,8] + 0.14583333333333334 y[2,13] + 0.14583333333333334 y[2,14] + 0.14583333333333334 y[2,15] + 0.14583333333333334 y[2,16] + 0.16666666666666666 y[3,3] + 0.16666666666666666 y[

-1222.1944433709298

In [170]:
@show m[:obj_first_stage_grad]
JuMP.value(m[:obj_first_stage_grad])

m[:obj_first_stage_grad] = 4


4.0

In [171]:
@show m[:obj_second_stage_grad]
JuMP.value(m[:obj_second_stage_grad])

m[:obj_second_stage_grad] = 0.015625 y[1,9] + 0.015625 y[1,10] + 0.015625 y[1,11] + 0.015625 y[1,12] + 0.015625 y[1,13] + 0.015625 y[1,14] + 0.015625 y[1,15] + 0.015625 y[1,16] + 0.015625 y[2,5] + 0.015625 y[2,6] + 0.015625 y[2,7] + 0.015625 y[2,8] + 0.015625 y[2,13] + 0.015625 y[2,14] + 0.015625 y[2,15] + 0.015625 y[2,16] + 0.015625 y[3,3] + 0.015625 y[3,4] + 0.015625 y[3,7] + 0.015625 y[3,8] + 0.015625 y[3,11] + 0.015625 y[3,12] + 0.015625 y[3,15] + 0.015625 y[3,16] + 0.015625 y[4,2] + 0.015625 y[4,4] + 0.015625 y[4,6] + 0.015625 y[4,8] + 0.015625 y[4,10] + 0.015625 y[4,12] + 0.015625 y[4,14] + 0.015625 y[4,16] + 6.25


5.125000002046381

In [172]:
for c in JuMP.all_constraints(m, include_variable_in_set_constraints=false)
    println(c)
    println(JuMP.dual(c))
end

x[1] + y[1,1] + y[2,1] + y[3,1] + y[4,1] = 8
0.6249999997953618
x[1] + y[1,2] + y[2,2] + y[3,2] + y[4,2] = 8
0.6249999997248551
x[1] + y[1,3] + y[2,3] + y[3,3] + y[4,3] = 8
0.6249999997248551
x[1] + y[1,4] + y[2,4] + y[3,4] + y[4,4] = 8
0.6249999995802237
x[1] + y[1,5] + y[2,5] + y[3,5] + y[4,5] = 8
0.6249999997248551
x[1] + y[1,6] + y[2,6] + y[3,6] + y[4,6] = 8
0.6249999995802236
x[1] + y[1,7] + y[2,7] + y[3,7] + y[4,7] = 8
0.6249999995802236
x[1] + y[1,8] + y[2,8] + y[3,8] + y[4,8] = 8
0.6249999991150602
x[1] + y[1,9] + y[2,9] + y[3,9] + y[4,9] = 8
0.624999999724855
x[1] + y[1,10] + y[2,10] + y[3,10] + y[4,10] = 8
0.6249999995802236
x[1] + y[1,11] + y[2,11] + y[3,11] + y[4,11] = 8
0.6249999995802235
x[1] + y[1,12] + y[2,12] + y[3,12] + y[4,12] = 8
0.6249999991150594
x[1] + y[1,13] + y[2,13] + y[3,13] + y[4,13] = 8
0.6249999995802235
x[1] + y[1,14] + y[2,14] + y[3,14] + y[4,14] = 8
0.6249999991150588
x[1] + y[1,15] + y[2,15] + y[3,15] + y[4,15] = 8
0.624999999115058
x[1] + y[1,16] + y

In [173]:
sum(ps .* JuMP.dual.(JuMP.all_constraints(m, include_variable_in_set_constraints=false)))

0.2500000000511595

In [174]:
bd_duals = JuMP.dual.(JuMP.all_constraints(m, include_variable_in_set_constraints=true))[end-ny*ns+1:end]
@show LA.norm(bd_duals, 2)
@show LA.norm(bd_duals, Inf)

LoadError: BoundsError: attempt to access 16-element Vector{Float64} at index [-47:16]

In [175]:
JuMP.all_constraints(m, include_variable_in_set_constraints=true) |> length

16

In [176]:
81 - 16

65

In [177]:
ns * ny

64

In [178]:
nc = 1 # number of constraints in scenario
H = LA.diagm(cy2)
W = ones(nc, ny)
A22 = zeros(nc, nc)
B1 = zeros(ny, nx)
T = ones(nc, nx)
A = [H W'; W A22]
B = [B1; -T]
Z = A \ B
Y = Z[1:ny, 1:nx]
Lambda = Z[ny+1:ny+nc, 1:nx]

1×1 Matrix{Float64}:
 0.25

In [179]:
Y

4×1 Matrix{Float64}:
 -0.25
 -0.25
 -0.25
 -0.25

In [180]:
"""
    wind_scenario_gradient(j, y_val, xi_val, cy, cr) -> Float64

Per-turbine marginal cost dQ/dy_j for one scenario.
Mirrors CudaqQAEOptimizer._wind_scenario_gradient in cudaq_impl.py.

  - Quadratic cost: cy[j,1] + 2*cy[j,2]*y_val  (when xi_val == 1)
  - Recourse cost:  cr                           (when xi_val == 0)
"""
function wind_scenario_gradient(j::Int, y_val::Real, xi_val::Real,
                                 cy::AbstractMatrix, cr::Real)
    if xi_val == 1
        return cy[j, 1] + 2.0 * cy[j, 2] * y_val
    else
        return cr
    end
end

"""
    estimate_expected_gradient(y_mat, xi_mat, ps, cy, cr) -> Vector{Float64}

Expected per-turbine gradient E[dQ/dy_j] over all scenarios.
Mirrors CudaqQAEOptimizer.estimate_expected_gradient_sv in cudaq_impl.py,
replacing DQA statevector sampling with the JuMP optimal y values.

Returns a vector of length ny; mean(result) matches phi_grad_cache[w_d].
"""
function estimate_expected_gradient(y_mat::AbstractMatrix, xi_mat::AbstractMatrix,
                                     ps::AbstractVector, cy::AbstractMatrix, cr::Real)
    ny, ns = size(xi_mat)
    grad = zeros(ny)
    for j in 1:ny
        for s in 1:ns
            grad[j] += ps[s] * wind_scenario_gradient(j, y_mat[j, s], xi_mat[j, s], cy, cr)
        end
    end
    return grad
end

# Evaluate at JuMP optimum and compare to the model expression
grad_vec  = estimate_expected_gradient(JuMP.value.(y), xi, ps, cy, cr)
phi_grad  = sum(grad_vec) / ny   # mean over turbines — matches phi_grad_cache[w_d]

@show grad_vec
@show phi_grad
@show Y' * grad_vec
@show JuMP.value(m[:obj_second_stage_grad])

grad_vec = [4.0000000040927635, 4.0000000040927635, 4.0000000040927635, 4.000000004092765]
phi_grad = 4.0000000040927635
Y' * grad_vec = [-4.0000000040927635]
JuMP.value(m[:obj_second_stage_grad]) = 5.125000002046381


5.125000002046381